> 在线讲义：[第 15 章 · 多轮交互与 Agent 任务：前缀、成功成本、关键路径](https://wikieden.github.io/ai-infra-from-constraints/lessons/15-agent-tasks.html) · 本书《从约束推导 AI Infra》

# 第 15 章 · 多轮交互与 Agent 任务：前缀、成功成本、关键路径

配套讲义：仓库里 `ai-infra/notes/lessons/15-agent-tasks.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 改写位置 | 第一个不同的 token 之前 | `reusable_prefix(old, new)` | 追加 100 → 2000；改第 501 个 → 500 |
| 第二步 价格例子 | $20 \times 0.1 \times 0.5 / 0.8$ | `success_cost_ratio(...)` | 1.25 |
| 第三步 例题 3-2 | Amdahl | `amdahl_task(76.510, 36.375, 2)` | 58.323 s，1.31 倍 |
| 第四步 工具 | 串行和 / 并行最大 | `task_time([2, (6, 10), 3], parallel=...)` | 21 s、15 s |
| 第五步 CED | $40P$ vs $20P + 20\min(P,128)$ | `ced_work(8192)` | 327,680 vs 166,400（50.8%） |
| 第五步 转交阈值 | $c_{\mathrm{dec}} / c_{\mathrm{LLM}}$ | `handoff_breakeven()` | 约 0.58% |

例题 3-2、工具 2/6/10/3 秒、CED 的 8192 个 token、决策请求费用取自 AII 第 3 章 §3.2；token 序列是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `reusable_prefix`**：最长可直接复用的前缀：第一个不同的 token 之前。

In [ ]:
def reusable_prefix(old: list[int], new: list[int]) -> int:
    """最长可直接复用的前缀：第一个不同的 token 之前。"""
    n = 0
    for a, b in zip(old, new):
        if a != b:
            break  # 第一个不同的位置之后全部失效
        n += 1
    return n

**函数 `success_cost`**：每个成功任务的期望成本 c / p。

In [ ]:
def success_cost(c: float, p: float) -> float:
    """每个成功任务的期望成本 c / p。"""
    return c / p

**函数 `success_cost_ratio`**：新旧方法每个成功任务的成本比：单价比 × 用量比 × (旧成功率 / 新成功率)。

In [ ]:
def success_cost_ratio(price: float, usage: float, p_old: float, p_new: float) -> float:
    """新旧方法每个成功任务的成本比：单价比 × 用量比 × (旧成功率 / 新成功率)。"""
    return price * usage * p_old / p_new

**函数 `amdahl_task`**：把占 part 秒的一段加速 s 倍：(新总时间, 总加速比)。

In [ ]:
def amdahl_task(total: float, part: float, s: float) -> tuple[float, float]:
    """把占 part 秒的一段加速 s 倍：(新总时间, 总加速比)。"""
    new = total - part + part / s
    return new, total / new

**函数 `task_time`**：模型 → 一组工具 → 模型：工具串行相加，独立时取最长。

In [ ]:
def task_time(pre: float, tools: tuple[float, ...], post: float, parallel: bool) -> float:
    """模型 → 一组工具 → 模型：工具串行相加，独立时取最长。"""
    return pre + (max(tools) if parallel else sum(tools)) + post

**函数 `ced_work`**：从空状态输入 P 个 token 的专家层工作：(普通 40 层, CED)。

In [ ]:
def ced_work(P: int) -> tuple[int, int]:
    """从空状态输入 P 个 token 的专家层工作：(普通 40 层, CED)。"""
    return 40 * P, 20 * P + 20 * min(P, 128)

**函数 `handoff_breakeven`**：转交比例超过多少时，转交 LLM 的费用超过决策模型本身。

In [ ]:
def handoff_breakeven(c_dec: float = 0.000081, c_llm: float = 0.013880) -> float:
    """转交比例超过多少时，转交 LLM 的费用超过决策模型本身。"""
    return c_dec / c_llm

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：改写位置决定可复用前缀（编的 2000 个 token）")
old = list(range(2000))
print(f"  末尾追加 100 个 → 复用 {reusable_prefix(old, old + list(range(100)))}")
changed = old.copy()
changed[500] = -1  # 改第 501 个
print(f"  改第 501 个 → 复用 {reusable_prefix(old, changed)}")

In [ ]:
print("\n第二步：单价 1/10、用量 20 倍、成功率 50% → 80%")
print(f"  成功任务成本比 = {success_cost_ratio(0.1, 20, 0.5, 0.8):.2f}")

In [ ]:
print("\n第三步：例题 3-2，首轮 36.375 s 减半")
new, sp = amdahl_task(76.510, 36.375, 2)
print(f"  76.510 → {new:.3f} s，加速 {sp:.2f} 倍")

In [ ]:
print("\n第四步：模型 2 s → 工具 6 s、10 s → 模型 3 s")
print(f"  串行 {task_time(2, (6, 10), 3, False):.0f} s，并行 {task_time(2, (6, 10), 3, True):.0f} s")

In [ ]:
print("\n第五步：CED 与决策请求")
full, ced = ced_work(8192)
print(f"  8192 个输入：普通 {full:,}，CED {ced:,}（{ced / full:.1%}）")
print(f"  转交比例超过 {handoff_breakeven():.2%} 时，转交费用超过决策模型本身")

## 3. 代码里才看得到的

1. `reusable_prefix` 逐个比 token：改动之后哪怕内容一样，位置变了也算不同（真实缓存按 token 序列的哈希块判断，原书第 8 章）。
2. 例题 3-2 用的总时间 76.510 秒包含控制和交接时间，比模型 + 工具的 76.372 秒多一点。
3. `task_time` 只处理"一段并行工具夹在两段模型之间"这种简单图；一般的依赖图要按拓扑顺序算最长路径。

## 4. 自己改一改

- 把首轮加速从 2 倍改成 10 倍、无穷大，看总加速比的上限。
- 用 `ced_work` 扫 $P$ = 64、128、512、8192、131072，看 CED 的节省比例怎么变。
- 把转交比例 $e$ 设成 1%、5%，算每条请求的平均费用。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 前情提要记得住，新内容才要算；改中间一个字，后面全重来；
> 驾照算总账，花费除以通过率；
> 修车师傅想得久，助手拧螺丝几秒钟；
> 烧水切菜可并行，工作不少灶台全占。